## Lab 5 - Evolutionary Algorithm for Knapsack

### 1) Goal
Implement an **Evolutionary Algorithm** (EA) for the 0/1 Knapsack problem. 
Requirements:
- Population initialization for binary codification
- Crossover operator
- Mutation operator
- Fitness function

Test the algorithm for the two problem instances (size 20 and 200), considering different parameter settings.

In [4]:
import math
import random
import time
from statistics import mean


def read_knapsack(filename):
    """Reads instance format used in previous labs."""
    items = []
    capacity = 0
    with open(filename, "r") as f:
        lines = [line.strip() for line in f if line.strip()]
    n = int(lines[0])
    for i in range(1, n + 1):
        parts = lines[i].split()
        weight = int(parts[1])
        value = int(parts[2])
        items.append((weight, value))
    capacity = int(lines[n + 1])
    return items, capacity


def evaluate_knapsack(solution, items, capacity):
    total_weight = sum(items[i][0] for i, bit in enumerate(solution) if bit)
    total_value = sum(items[i][1] for i, bit in enumerate(solution) if bit)
    if total_weight > capacity:
        return 0
    return total_value


def initial_greedy_knapsack(items, capacity):
    """Value/weight greedy baseline for comparison."""
    n = len(items)
    order = sorted(range(n), key=lambda i: items[i][1] / items[i][0], reverse=True)
    sol = [0] * n
    used = 0
    for i in order:
        w, _ = items[i]
        if used + w <= capacity:
            sol[i] = 1
            used += w
    return sol

In [5]:
def init_population(pop_size, n):
    """Initializes population with random binary strings."""
    return [[random.randint(0, 1) for _ in range(n)] for _ in range(pop_size)]


def select_parent_tournament(population, fitnesses, tournament_size=3):
    """Selects one parent using tournament selection."""
    indices = random.sample(range(len(population)), tournament_size)
    best_idx = max(indices, key=lambda i: fitnesses[i])
    return population[best_idx]


def crossover_1point(p1, p2):
    """Performs 1-point crossover between two parents."""
    n = len(p1)
    if n <= 1:
        return p1[:], p2[:]
    pt = random.randint(1, n - 1)
    o1 = p1[:pt] + p2[pt:]
    o2 = p2[:pt] + p1[pt:]
    return o1, o2


def mutate_bitflip(solution, mutation_rate):
    """Flips bits in the solution based on mutation_rate."""
    return [1 - bit if random.random() < mutation_rate else bit for bit in solution]

In [6]:
def run_ea_knapsack(
    items,
    capacity,
    pop_size=100,
    generations=100,
    tournament_size=3,
    crossover_rate=0.8,
    mutation_rate=0.01,
    elitism=True,
    seed=None,
):
    """Runs the Evolutionary Algorithm for the Knapsack problem."""
    if seed is not None:
        random.seed(seed)

    n = len(items)
    population = init_population(pop_size, n)
    fitnesses = [evaluate_knapsack(ind, items, capacity) for ind in population]

    best_idx = max(range(pop_size), key=lambda i: fitnesses[i])
    global_best_sol = population[best_idx][:]
    global_best_fit = fitnesses[best_idx]

    history = []

    for gen in range(generations):
        new_population = []

        if elitism:
            new_population.append(global_best_sol[:])

        while len(new_population) < pop_size:
            p1 = select_parent_tournament(population, fitnesses, tournament_size)
            p2 = select_parent_tournament(population, fitnesses, tournament_size)

            if random.random() < crossover_rate:
                o1, o2 = crossover_1point(p1, p2)
            else:
                o1, o2 = p1[:], p2[:]

            o1 = mutate_bitflip(o1, mutation_rate)
            o2 = mutate_bitflip(o2, mutation_rate)

            new_population.append(o1)
            if len(new_population) < pop_size:
                new_population.append(o2)

        population = new_population
        fitnesses = [evaluate_knapsack(ind, items, capacity) for ind in population]

        curr_best_idx = max(range(pop_size), key=lambda i: fitnesses[i])
        if fitnesses[curr_best_idx] > global_best_fit:
            global_best_fit = fitnesses[curr_best_idx]
            global_best_sol = population[curr_best_idx][:]

        if gen % max(1, generations // 10) == 0 or gen == generations - 1:
            history.append((gen, global_best_fit, mean(fitnesses)))

    return {
        "best_solution": global_best_sol,
        "best_value": global_best_fit,
        "history": history,
    }

In [7]:
def run_ea_experiments(instance_file, parameter_grid, num_runs=5):
    items, capacity = read_knapsack(instance_file)
    n = len(items)
    output_file = f"results_ea_knapsack_n{n}.txt"

    greedy_sol = initial_greedy_knapsack(items, capacity)
    greedy_value = evaluate_knapsack(greedy_sol, items, capacity)

    rows = []

    with open(output_file, "w") as f:
        f.write(f"--- EA Knapsack Results for {instance_file} ---\n")
        f.write(f"n={n}, capacity={capacity}, runs_per_setting={num_runs}\n")
        f.write(f"Greedy baseline value={greedy_value}\n")

        for idx, params in enumerate(parameter_grid, start=1):
            pop_size = params["pop_size"]
            generations = params["generations"]
            tournament_size = params.get("tournament_size", 3)
            crossover_rate = params.get("crossover_rate", 0.8)
            mutation_rate = params.get("mutation_rate", 1.0 / n)
            elitism = params.get("elitism", True)

            values = []
            times = []

            print(
                f"[n={n}] setting {idx}/{len(parameter_grid)}: "
                f"pop={pop_size}, gen={generations}, mut={mutation_rate:.3f}, cross={crossover_rate:.2f}"
            )

            f.write("\n" + "=" * 70 + "\n")
            f.write(
                f"Setting {idx}: pop_size={pop_size}, gen={generations}, tourn_size={tournament_size}, "
                f"cross_rate={crossover_rate}, mut_rate={mutation_rate:.3f}, elitism={elitism}\n"
            )
            f.write("-" * 70 + "\n")

            for run_idx in range(num_runs):
                seed = 8000 + run_idx
                t0 = time.time()
                result = run_ea_knapsack(
                    items,
                    capacity,
                    pop_size=pop_size,
                    generations=generations,
                    tournament_size=tournament_size,
                    crossover_rate=crossover_rate,
                    mutation_rate=mutation_rate,
                    elitism=elitism,
                    seed=seed,
                )
                elapsed = time.time() - t0

                values.append(result["best_value"])
                times.append(elapsed)

                f.write(
                    f"Run {run_idx + 1}: value={result['best_value']}, "
                    f"time={elapsed:.4f}s\n"
                )

            avg_val = mean(values)
            best_val = max(values)
            worst_val = min(values)
            avg_time = mean(times)
            improve = 100.0 * (avg_val - greedy_value) / max(greedy_value, 1)

            f.write(
                f">> avg={avg_val:.2f}, best={best_val}, worst={worst_val}, "
                f"avg_time={avg_time:.4f}s, improve_vs_greedy={improve:.2f}%\n"
            )

            rows.append(
                {
                    "n": n,
                    "pop_size": pop_size,
                    "generations": generations,
                    "tournament_size": tournament_size,
                    "crossover_rate": crossover_rate,
                    "mutation_rate": mutation_rate,
                    "avg": avg_val,
                    "best": best_val,
                    "worst": worst_val,
                    "avg_time": avg_time,
                    "improve_pct": improve,
                }
            )

    print(f"Done: {output_file}")
    return rows

In [ ]:
# Define parameter grids for both instances
knapsack_20_grid = [
    {"pop_size": 50, "generations": 100, "crossover_rate": 0.8, "mutation_rate": 0.05},
    {"pop_size": 100, "generations": 200, "crossover_rate": 0.9, "mutation_rate": 0.01},
    {"pop_size": 50, "generations": 100, "crossover_rate": 0.7, "mutation_rate": 0.10},
    {"pop_size": 200, "generations": 150, "crossover_rate": 0.85, "mutation_rate": 0.05},
]

knapsack_200_grid = [
    {"pop_size": 100, "generations": 300, "crossover_rate": 0.8, "mutation_rate": 0.005},
    {"pop_size": 200, "generations": 500, "crossover_rate": 0.9, "mutation_rate": 0.01},
    {"pop_size": 100, "generations": 300, "crossover_rate": 0.7, "mutation_rate": 0.05},
    {"pop_size": 300, "generations": 600, "crossover_rate": 0.85, "mutation_rate": 0.01},
]


# Run the comparative experiments
#rows20 = run_ea_experiments("../data/knapsack-20.txt", knapsack_20_grid, num_runs=5)
#rows200 = run_ea_experiments("../data/knapsack-200.txt", knapsack_200_grid, num_runs=3)

# Display the summary of parameters and results
#all_rows = rows20 + rows200
#print("\nSummary by instance (sorted by average value desc):")
# for n in [20, 200]:
#     subset = [r for r in all_rows if r["n"] == n]
#     subset = sorted(subset, key=lambda r: r["avg"], reverse=True)
#     print(f"\nInstance n={n}")
#     for r in subset:
#         print(
#             f"pop={r['pop_size']:3d} | gen={r['generations']:4d} | "
#             f"cross={r['crossover_rate']:.2f} | mut={r['mutation_rate']:.3f} | "
#             f"avg={r['avg']:.2f} | best={r['best']} | "
#             f"impr={r['improve_pct']:.2f}% | t={r['avg_time']:.3f}s"
#         )

[n=20] setting 1/4: pop=50, gen=100, mut=0.050, cross=0.80
[n=20] setting 2/4: pop=100, gen=200, mut=0.010, cross=0.90
[n=20] setting 3/4: pop=50, gen=100, mut=0.100, cross=0.70
[n=20] setting 4/4: pop=200, gen=150, mut=0.050, cross=0.85
Done: results_ea_knapsack_n20.txt
[n=200] setting 1/4: pop=100, gen=300, mut=0.005, cross=0.80
[n=200] setting 2/4: pop=200, gen=500, mut=0.010, cross=0.90
[n=200] setting 3/4: pop=100, gen=300, mut=0.050, cross=0.70
[n=200] setting 4/4: pop=300, gen=600, mut=0.010, cross=0.85
Done: results_ea_knapsack_n200.txt

Summary by instance (sorted by average value desc):

Instance n=20
pop=200 | gen= 150 | cross=0.85 | mut=0.050 | avg=787.00 | best=787 | impr=0.00% | t=0.392s
pop= 50 | gen= 100 | cross=0.80 | mut=0.050 | avg=786.20 | best=787 | impr=-0.10% | t=0.055s
pop=100 | gen= 200 | cross=0.90 | mut=0.010 | avg=786.20 | best=787 | impr=-0.10% | t=0.233s
pop= 50 | gen= 100 | cross=0.70 | mut=0.100 | avg=786.20 | best=787 | impr=-0.10% | t=0.062s

Instance 

## Assignment A5 - Evolutionary Algorithm for TSP

### 1) Problem Definition
The Traveling Salesman Problem (TSP) asks for a minimum-length closed tour that visits each city exactly once and returns to the start city.

### 2) EA Representation and Operators
We use a permutation representation with city `0` fixed at index `0`.

- **Individual**: route permutation `[0, ...]`
- **Fitness**: inverse of tour length (`1 / length`) so that larger is better
- **Selection**: tournament selection
- **Mutation**: `swap` and `inversion`
- **Crossover** (for comparison):
  - `OX` (Order Crossover)
  - `PMX` (Partially Mapped Crossover)

### 3) Pseudocode
```text
Input: coords, pop_size, generations, crossover_type, mutation_type
P <- initialize population (greedy + random tours)
evaluate P
best <- best(P)

for gen = 1..generations:
    P_new <- elite(best) if elitism
    while |P_new| < pop_size:
        p1, p2 <- tournament_select(P)
        c1, c2 <- crossover(p1, p2) with probability pc
        c1 <- mutate(c1) with probability pm
        c2 <- mutate(c2) with probability pm
        add children to P_new
    P <- P_new
    evaluate P
    update best

return best
```

In [10]:
import glob
import os
import math
import random
import time
from statistics import mean


def read_tsp(file_path):
    """Reads a TSPLIB EUC_2D instance and returns name, dimension, coordinates."""
    name = None
    dimension = None
    in_coord_section = False
    coords_by_id = {}

    with open(file_path, "r") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line:
                continue

            if line.startswith("NAME"):
                name = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
            elif line.startswith("DIMENSION"):
                rhs = line.split(":", 1)[1].strip() if ":" in line else line.split()[-1]
                dimension = int(rhs)
            elif line == "NODE_COORD_SECTION":
                in_coord_section = True
            elif line == "EOF":
                break
            elif in_coord_section:
                parts = line.split()
                if len(parts) >= 3:
                    city_id = int(parts[0])
                    x = float(parts[1])
                    y = float(parts[2])
                    coords_by_id[city_id] = (x, y)

    ordered_ids = sorted(coords_by_id.keys())
    coords = [coords_by_id[i] for i in ordered_ids]

    if dimension is None:
        dimension = len(coords)

    return {
        "name": name if name else file_path,
        "dimension": dimension,
        "coords": coords,
    }


def euclidean_distance(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


def tour_length(route, coords):
    total = 0.0
    n = len(route)
    for i in range(n):
        c1 = coords[route[i]]
        c2 = coords[route[(i + 1) % n]]
        total += euclidean_distance(c1, c2)
    return total


def greedy_tour(coords, start=0):
    n = len(coords)
    unvisited = set(range(n))
    route = [start]
    unvisited.remove(start)
    current = start

    while unvisited:
        nxt = min(unvisited, key=lambda c: euclidean_distance(coords[current], coords[c]))
        route.append(nxt)
        unvisited.remove(nxt)
        current = nxt

    return route


def random_tour(n, start=0):
    nodes = list(range(n))
    nodes.remove(start)
    random.shuffle(nodes)
    return [start] + nodes


def init_population_tsp(pop_size, coords):
    """Hybrid init: one greedy seed + random tours for diversification."""
    n = len(coords)
    population = [greedy_tour(coords, start=0)]
    while len(population) < pop_size:
        population.append(random_tour(n, start=0))
    return population


def fitness_from_length(length):
    return 1.0 / max(length, 1e-12)


def select_parent_tournament(population, fitnesses, tournament_size=3):
    indices = random.sample(range(len(population)), tournament_size)
    best_idx = max(indices, key=lambda i: fitnesses[i])
    return population[best_idx]


def crossover_ox(parent1, parent2):
    """Order Crossover (OX) with city 0 fixed at the first position."""
    n = len(parent1)
    if n <= 3:
        return parent1[:], parent2[:]

    p1 = parent1[1:]
    p2 = parent2[1:]
    m = len(p1)

    i = random.randint(0, m - 2)
    j = random.randint(i + 1, m - 1)

    c1 = [None] * m
    c2 = [None] * m
    c1[i : j + 1] = p1[i : j + 1]
    c2[i : j + 1] = p2[i : j + 1]

    fill_p2 = [x for x in p2 if x not in c1]
    fill_p1 = [x for x in p1 if x not in c2]

    pos1 = [k for k in range(m) if c1[k] is None]
    pos2 = [k for k in range(m) if c2[k] is None]

    for k, city in zip(pos1, fill_p2):
        c1[k] = city
    for k, city in zip(pos2, fill_p1):
        c2[k] = city

    return [0] + c1, [0] + c2


def _pmx_resolve(value, segment_set, inverse_parent, mapping):
    """Resolve mapping chain safely; breaks potential cycles in malformed chains."""
    seen = set()
    while value in segment_set and value not in seen:
        seen.add(value)
        idx = inverse_parent[value]
        value = mapping[idx]
    return value


def crossover_pmx(parent1, parent2):
    """Partially Mapped Crossover (PMX) with city 0 fixed at index 0."""
    n = len(parent1)
    if n <= 3:
        return parent1[:], parent2[:]

    p1 = parent1[1:]
    p2 = parent2[1:]
    m = len(p1)

    i = random.randint(0, m - 2)
    j = random.randint(i + 1, m - 1)

    c1 = [None] * m
    c2 = [None] * m

    c1[i : j + 1] = p1[i : j + 1]
    c2[i : j + 1] = p2[i : j + 1]

    seg1 = set(p1[i : j + 1])
    seg2 = set(p2[i : j + 1])

    inv1 = {city: idx for idx, city in enumerate(p1)}
    inv2 = {city: idx for idx, city in enumerate(p2)}

    for k in range(m):
        if i <= k <= j:
            continue

        v1 = p2[k]
        if v1 in seg1:
            v1 = _pmx_resolve(v1, seg1, inv2, p1)
        c1[k] = v1

        v2 = p1[k]
        if v2 in seg2:
            v2 = _pmx_resolve(v2, seg2, inv1, p2)
        c2[k] = v2

    return [0] + c1, [0] + c2


def mutate_swap(route, mutation_rate):
    if random.random() >= mutation_rate or len(route) <= 3:
        return route[:]
    out = route[:]
    i = random.randint(1, len(route) - 2)
    j = random.randint(i + 1, len(route) - 1)
    out[i], out[j] = out[j], out[i]
    return out


def mutate_inversion(route, mutation_rate):
    if random.random() >= mutation_rate or len(route) <= 3:
        return route[:]
    out = route[:]
    i = random.randint(1, len(route) - 2)
    j = random.randint(i + 1, len(route) - 1)
    out[i : j + 1] = reversed(out[i : j + 1])
    return out


def mutate_route(route, mutation_rate, mutation_type):
    if mutation_type == "swap":
        return mutate_swap(route, mutation_rate)
    if mutation_type == "inversion":
        return mutate_inversion(route, mutation_rate)
    raise ValueError(f"Unknown mutation_type: {mutation_type}")

In [13]:
def crossover_pmx(parent1, parent2):
    """Safe PMX variant (DEAP-style) with city 0 fixed at index 0."""
    n = len(parent1)
    if n <= 3:
        return parent1[:], parent2[:]

    p1 = parent1[1:]
    p2 = parent2[1:]
    m = len(p1)

    cx1 = random.randint(0, m - 2)
    cx2 = random.randint(cx1 + 1, m - 1)

    c1 = p1[:]
    c2 = p2[:]

    for i in range(cx1, cx2 + 1):
        g1 = c1[i]
        g2 = c2[i]

        j1 = c1.index(g2)
        j2 = c2.index(g1)

        c1[i], c1[j1] = c1[j1], c1[i]
        c2[i], c2[j2] = c2[j2], c2[i]

    return [0] + c1, [0] + c2

In [14]:
def run_ea_tsp(
    coords,
    pop_size=80,
    generations=200,
    tournament_size=3,
    crossover_rate=0.9,
    mutation_rate=0.15,
    crossover_type="ox",
    mutation_type="inversion",
    elitism=True,
    seed=None,
):
    """Evolutionary Algorithm for TSP (minimization handled via inverse-length fitness)."""
    if seed is not None:
        random.seed(seed)

    population = init_population_tsp(pop_size, coords)
    lengths = [tour_length(ind, coords) for ind in population]
    fitnesses = [fitness_from_length(v) for v in lengths]

    best_idx = min(range(pop_size), key=lambda i: lengths[i])
    global_best_tour = population[best_idx][:]
    global_best_length = lengths[best_idx]

    history = []

    for gen in range(generations):
        new_population = []

        if elitism:
            new_population.append(global_best_tour[:])

        while len(new_population) < pop_size:
            p1 = select_parent_tournament(population, fitnesses, tournament_size)
            p2 = select_parent_tournament(population, fitnesses, tournament_size)

            if random.random() < crossover_rate:
                if crossover_type == "ox":
                    c1, c2 = crossover_ox(p1, p2)
                elif crossover_type == "pmx":
                    c1, c2 = crossover_pmx(p1, p2)
                else:
                    raise ValueError(f"Unknown crossover_type: {crossover_type}")
            else:
                c1, c2 = p1[:], p2[:]

            c1 = mutate_route(c1, mutation_rate, mutation_type)
            c2 = mutate_route(c2, mutation_rate, mutation_type)

            new_population.append(c1)
            if len(new_population) < pop_size:
                new_population.append(c2)

        population = new_population
        lengths = [tour_length(ind, coords) for ind in population]
        fitnesses = [fitness_from_length(v) for v in lengths]

        curr_best_idx = min(range(pop_size), key=lambda i: lengths[i])
        if lengths[curr_best_idx] < global_best_length:
            global_best_length = lengths[curr_best_idx]
            global_best_tour = population[curr_best_idx][:]

        if gen % max(1, generations // 10) == 0 or gen == generations - 1:
            history.append((gen, global_best_length, mean(lengths)))

    return {
        "best_tour": global_best_tour,
        "best_length": global_best_length,
        "history": history,
    }


def run_ea_tsp_experiments(instance_paths, parameter_grid, num_runs=3, output_filename="results_ea_tsp.txt"):
    rows = []

    with open(output_filename, "w") as f:
        f.write("--- Evolutionary Algorithm TSP Results ---\n")
        f.write(f"Runs per parameter setting: {num_runs}\n")

        for instance_path in instance_paths:
            tsp = read_tsp(instance_path)
            coords = tsp["coords"]
            n = tsp["dimension"]

            print(f"\n[Instance] {tsp['name']} (n={n})")
            f.write("\n" + "=" * 72 + "\n")
            f.write(f"Instance: {tsp['name']} | n={n}\n")
            f.write("=" * 72 + "\n")

            greedy = greedy_tour(coords, start=0)
            greedy_length = tour_length(greedy, coords)
            print(f"  Greedy baseline: {greedy_length:.2f}")
            f.write(f"Greedy baseline length: {greedy_length:.2f}\n")

            for idx, params in enumerate(parameter_grid, start=1):
                pop_size = params["pop_size"]
                generations = params["generations"]
                tournament_size = params.get("tournament_size", 3)
                crossover_rate = params.get("crossover_rate", 0.9)
                mutation_rate = params.get("mutation_rate", 0.15)
                crossover_type = params.get("crossover_type", "ox")
                mutation_type = params.get("mutation_type", "inversion")
                elitism = params.get("elitism", True)

                lengths = []
                times = []

                print(
                    f"  [Setting {idx}/{len(parameter_grid)}] "
                    f"cross={crossover_type}, pop={pop_size}, gen={generations}, "
                    f"mut={mutation_type}/{mutation_rate:.3f}, cross_rate={crossover_rate:.2f}"
                )
                f.write(
                    f"\nParams: crossover={crossover_type}, pop_size={pop_size}, generations={generations}, "
                    f"tournament={tournament_size}, crossover_rate={crossover_rate}, mutation_type={mutation_type}, "
                    f"mutation_rate={mutation_rate:.4f}, elitism={elitism}\n"
                )
                f.write("-" * 72 + "\n")

                for run_idx in range(num_runs):
                    seed = 9000 + run_idx
                    t0 = time.time()
                    result = run_ea_tsp(
                        coords,
                        pop_size=pop_size,
                        generations=generations,
                        tournament_size=tournament_size,
                        crossover_rate=crossover_rate,
                        mutation_rate=mutation_rate,
                        crossover_type=crossover_type,
                        mutation_type=mutation_type,
                        elitism=elitism,
                        seed=seed,
                    )
                    elapsed = time.time() - t0

                    best_length = result["best_length"]
                    best_tour = result["best_tour"]
                    lengths.append(best_length)
                    times.append(elapsed)

                    print(f"    Run {run_idx + 1}/{num_runs}: length={best_length:.2f}, time={elapsed:.2f}s")
                    f.write(
                        f"Run {run_idx + 1}: length={best_length:.2f}, "
                        f"time={elapsed:.4f}s, tour_prefix={best_tour[:10]}\n"
                    )

                avg_len = mean(lengths)
                best_len = min(lengths)
                worst_len = max(lengths)
                avg_time = mean(times)
                improvement = 100.0 * (greedy_length - avg_len) / max(greedy_length, 1e-12)

                print(
                    f"    Summary: avg={avg_len:.2f}, best={best_len:.2f}, "
                    f"improvement={improvement:.2f}%, avg_time={avg_time:.2f}s"
                )
                f.write(
                    f">> Avg length={avg_len:.2f} | Best={best_len:.2f} | Worst={worst_len:.2f} | "
                    f"Avg time={avg_time:.4f}s | Avg improvement vs greedy={improvement:.2f}%\n"
                )

                rows.append(
                    {
                        "instance": tsp["name"],
                        "n": n,
                        "crossover_type": crossover_type,
                        "mutation_type": mutation_type,
                        "pop_size": pop_size,
                        "generations": generations,
                        "crossover_rate": crossover_rate,
                        "mutation_rate": mutation_rate,
                        "avg_length": avg_len,
                        "best_length": best_len,
                        "avg_time": avg_time,
                        "avg_improvement_pct": improvement,
                    }
                )

    print(f"\nDone! Results saved to {output_filename}")
    return rows


def run_ea_tsp_all_instances(
    data_root="../data",
    pop_size=30,
    generations=60,
    num_runs=1,
    output_filename="results_ea_tsp_all_instances.txt",
):
    """
    Runs a lightweight EA benchmark on all available TSP instances.
    The budget is intentionally small to keep all-instance execution tractable.
    """
    instance_paths = sorted(glob.glob(os.path.join(data_root, "A", "*.tsp")))
    instance_paths += sorted(glob.glob(os.path.join(data_root, "B", "*.tsp")))

    rows = []

    with open(output_filename, "w") as f:
        f.write("--- EA TSP Results (All Available Instances) ---\n")
        f.write(f"Instances={len(instance_paths)}, runs_per_instance={num_runs}\n")
        f.write(
            f"Common params: pop_size={pop_size}, generations={generations}, "
            f"crossover=ox, mutation=inversion\n"
        )

        for idx, path in enumerate(instance_paths, start=1):
            tsp = read_tsp(path)
            coords = tsp["coords"]
            n = tsp["dimension"]

            greedy = greedy_tour(coords, start=0)
            greedy_length = tour_length(greedy, coords)

            lengths = []
            times = []
            for run_idx in range(num_runs):
                seed = 12000 + run_idx
                t0 = time.time()
                result = run_ea_tsp(
                    coords,
                    pop_size=pop_size,
                    generations=generations,
                    crossover_type="ox",
                    mutation_type="inversion",
                    crossover_rate=0.9,
                    mutation_rate=0.15,
                    elitism=True,
                    seed=seed,
                )
                elapsed = time.time() - t0
                lengths.append(result["best_length"])
                times.append(elapsed)

            avg_len = mean(lengths)
            best_len = min(lengths)
            avg_time = mean(times)
            improvement = 100.0 * (greedy_length - avg_len) / max(greedy_length, 1e-12)

            print(f"[{idx}/{len(instance_paths)}] {tsp['name']} | n={n} | avg={avg_len:.2f} | best={best_len:.2f}")
            f.write(
                f"{tsp['name']:>12} | n={n:6d} | greedy={greedy_length:12.2f} | "
                f"avg={avg_len:12.2f} | best={best_len:12.2f} | "
                f"impr={improvement:7.2f}% | t={avg_time:8.3f}s\n"
            )

            rows.append(
                {
                    "instance": tsp["name"],
                    "n": n,
                    "greedy_length": greedy_length,
                    "avg_length": avg_len,
                    "best_length": best_len,
                    "avg_time": avg_time,
                    "avg_improvement_pct": improvement,
                }
            )

    print(f"Done! Results saved to {output_filename}")
    return rows

In [15]:
# 1) Two selected TSP instances (as in previous labs)
assignment_tsp_instances = [
    "../data/A/pr107.tsp",
    "../data/B/lu980.tsp",
]

# 2) Parameter grid with at least two crossover operators (OX and PMX)
assignment_ea_tsp_grid = [
    {
        "crossover_type": "ox",
        "mutation_type": "swap",
        "pop_size": 60,
        "generations": 120,
        "crossover_rate": 0.9,
        "mutation_rate": 0.10,
        "tournament_size": 3,
    },
    {
        "crossover_type": "ox",
        "mutation_type": "inversion",
        "pop_size": 80,
        "generations": 180,
        "crossover_rate": 0.9,
        "mutation_rate": 0.15,
        "tournament_size": 4,
    },
    {
        "crossover_type": "pmx",
        "mutation_type": "swap",
        "pop_size": 60,
        "generations": 120,
        "crossover_rate": 0.9,
        "mutation_rate": 0.10,
        "tournament_size": 3,
    },
    {
        "crossover_type": "pmx",
        "mutation_type": "inversion",
        "pop_size": 80,
        "generations": 180,
        "crossover_rate": 0.9,
        "mutation_rate": 0.15,
        "tournament_size": 4,
    },
]

# 3) Experiments for selected instances
ea_selected_rows = run_ea_tsp_experiments(
    assignment_tsp_instances,
    assignment_ea_tsp_grid,
    num_runs=3,
    output_filename="results_ea_tsp.txt",
)

print("\nTop settings on selected instances (sorted by avg length):")
for row in sorted(ea_selected_rows, key=lambda r: (r["instance"], r["avg_length"])):
    print(
        f"{row['instance']:>10} | n={row['n']:4d} | cross={row['crossover_type']:>3} | "
        f"mut={row['mutation_type']:>9} | pop={row['pop_size']:3d} | gen={row['generations']:4d} | "
        f"avg={row['avg_length']:.2f} | best={row['best_length']:.2f} | "
        f"impr={row['avg_improvement_pct']:.2f}% | t={row['avg_time']:.2f}s"
    )

print("\nSelected-instance experiments done. Run the next cell for all instances.")


[Instance] pr107 (n=107)
  Greedy baseline: 46678.15
  [Setting 1/4] cross=ox, pop=60, gen=120, mut=swap/0.100, cross_rate=0.90
    Run 1/3: length=46547.47, time=1.31s
    Run 2/3: length=46480.70, time=1.20s
    Run 3/3: length=46463.00, time=1.21s
    Summary: avg=46497.06, best=46463.00, improvement=0.39%, avg_time=1.24s
  [Setting 2/4] cross=ox, pop=80, gen=180, mut=inversion/0.150, cross_rate=0.90
    Run 1/3: length=45865.69, time=2.50s
    Run 2/3: length=45291.53, time=2.46s
    Run 3/3: length=45702.31, time=2.42s
    Summary: avg=45619.84, best=45291.53, improvement=2.27%, avg_time=2.46s
  [Setting 3/4] cross=pmx, pop=60, gen=120, mut=swap/0.100, cross_rate=0.90
    Run 1/3: length=46547.47, time=0.35s
    Run 2/3: length=46480.70, time=0.34s
    Run 3/3: length=46463.00, time=0.35s
    Summary: avg=46497.06, best=46463.00, improvement=0.39%, avg_time=0.35s
  [Setting 4/4] cross=pmx, pop=80, gen=180, mut=inversion/0.150, cross_rate=0.90
    Run 1/3: length=46100.00, time=0.

In [ ]:
# 4) Experiments on all available instances (separate cell, can be long)

all_instance_rows = run_ea_tsp_all_instances(
    data_root="../data",
    pop_size=20,
    generations=40,
    num_runs=1,
    output_filename="results_ea_tsp_all_instances.txt",
)

print("\nBest all-instance improvements:")
for row in sorted(all_instance_rows, key=lambda r: r["avg_improvement_pct"], reverse=True)[:10]:
    print(
        f"{row['instance']:>12} | n={row['n']:6d} | "
        f"avg={row['avg_length']:.2f} | best={row['best_length']:.2f} | "
        f"impr={row['avg_improvement_pct']:.2f}% | t={row['avg_time']:.2f}s"
    )

## A5 Report - Evolutionary Algorithm for TSP

### 1) Problem Definition
We solve TSP as a minimization problem: find a minimum-length Hamiltonian cycle.

### 2) Algorithm Used
Algorithm: Evolutionary Algorithm (EA) with permutation encoding.

Main steps:
1. Build initial population (`greedy` + random tours).
2. Evaluate each route using inverse-tour-length fitness.
3. Select parents with tournament selection.
4. Apply crossover (`OX` or `PMX`).
5. Apply mutation (`swap` or `inversion`).
6. Keep elite best solution (elitism).
7. Repeat for a fixed number of generations.

### 3) Parameter Settings
Selected-instance experiments use a grid varying:
- crossover operator: `OX` and `PMX`
- mutation operator: `swap`, `inversion`
- `pop_size`, `generations`, `mutation_rate`, `tournament_size`
- `num_runs = 3`

All-instance experiments are run with a lighter default budget to keep execution tractable across very large instances.

Output files:
- `results_ea_tsp.txt` (selected two instances)
- `results_ea_tsp_all_instances.txt` (all available TSP instances)

### 4) Comparative Results
Use the generated output files to summarize:
- quality (`avg length`, `best length`)
- parameter settings
- number of runs
- average improvement vs greedy baseline

For the crossover comparison, directly compare `OX` vs `PMX` rows under matched mutation and population settings.

### 5) Discussion
In the discussion include:
- which crossover operator was better overall (`OX` vs `PMX`),
- which parameter settings gave the best tradeoff quality/time,
- behavior differences between smaller and very large instances,
- limitations and possible improvements (larger budgets, local search hybridization, adaptive mutation).